# L6: Canvas Applications
In this lesson, you'll build a fullstack todo app where the agent and the frontend share the same live state. The agent writes todos from the backend; the user edits them in the UI — both sides stay in sync automatically.

## 📋 Learning Objectives

1. **Shared agent state** - Keep the backend and frontend in sync after every turn
2. **Backend tools with state updates** - Write tools that read and update state with `Command(update={...})`
3. **Frontend tools** - Trigger browser-side behavior from the agent with `useFrontendTool`

In [ ]:
# clear your state if running notebook more than once
from helper import reset_lesson
reset_lesson(6)

## Setup dependencies

In [ ]:
# install dependencies
# !pip install -r requirements.txt

In [ ]:
# load front end requirements
from helper import install_frontend
install_frontend()

In [ ]:
from helper import load_api_keys
load_api_keys()

## Start frontend and agent server
This cell stands up a FastAPI server that hosts a minimal LangGraph agent — no tools, generic prompt — and exposes it to the frontend over AG-UI. CopilotKitMiddleware enables shared state with the UI, MemorySaver persists state across turns, and add_langgraph_fastapi_endpoint mounts the agent at /. You'll extend the tools and system_prompt in the steps below.

In [ ]:
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from copilotkit import CopilotKitMiddleware, LangGraphAGUIAgent
from fastapi import FastAPI
from helper import start_server
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

app = FastAPI()

agent = LangGraphAGUIAgent(
    name="default",
    description="Lesson 6 shared-state todo agent",
    
    # We're going to update this agent in the steps below!
    graph=create_agent(
        model=ChatOpenAI(model="gpt-4.1"),
        tools=[],
        middleware=[CopilotKitMiddleware()],
        checkpointer=MemorySaver(),
        system_prompt="You are a helpful assistant.",
    ),
)

add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")
start_server(app, port=8006)

## What you'll build

A chat + todo panel where the agent can create, edit, and remove todos, and the user can check them off directly in the UI. Try this prompt once the app is running:

<div style="background: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 20px; padding: 8px 16px; display: inline-block; font-size: 14px; color: #333; margin: 8px 0;">Add three todos about learning CopilotKit</div>

In [ ]:
from helper import start_frontend
start_frontend(port=3006)

In [ ]:
from helper import display_app
display_app(port=3006)

## Phase 1: Shared state and backend tools

Shared agent state is a typed JSON object that both the backend agent and the frontend can read and write. CopilotKit synchronizes both sides after every turn.

You'll define a `Todo` schema and two tools:
- **`manage_todos`** — bulk-replaces the todo list (handles adds, edits, and removals)
- **`get_todos`** — reads the current list so the agent can inspect it before making changes

### Write the state schema and tools

Note: Unlike other cells in this lesson, we're running this code directly in the notebook rather than writing it to backend/todos.py. The agent server (started in the previous cell) is already running with a reference to the agent object in this kernel, so we can update its graph in place by running cell 16 below — no file write or server restart needed. The Todo, AgentState, and todo_tools defined here live in the notebook's namespace and get passed straight into create_agent().

In [ ]:
from __future__ import annotations

import uuid

from langchain.agents.middleware import AgentState as BaseAgentState
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from typing_extensions import TypedDict


class Todo(TypedDict):
    id: str
    title: str
    completed: bool


class AgentState(BaseAgentState):
    todos: list[Todo]

In [ ]:
@tool
def manage_todos(todos: list[Todo], runtime: ToolRuntime) -> Command:
    """Replace the entire todo list. Use this to add, edit, or remove todos."""
    for todo in todos:
        if not todo.get("id"):
            todo["id"] = str(uuid.uuid4())

    # Command allows agents to update state via toolcalls
    return Command(update={
        "todos": todos,
        "messages": [
            ToolMessage(
                content="Successfully updated todos",
                tool_call_id=runtime.tool_call_id,
            )
        ],
    })


@tool
def get_todos(runtime: ToolRuntime):
    """Get the current todo list."""
    return runtime.state.get("todos", [])


todo_tools = [manage_todos, get_todos]

### Update the agent to have tools and state

In [ ]:
from backend.todos import AgentState
from copilotkit import CopilotKitMiddleware
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

agent.graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    state_schema=AgentState,
    tools=todo_tools,
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=(
        "You manage a shared todo list. "
        "Use manage_todos to add, edit, or remove todos. "
        "Use get_todos to check the current list. "
        "When asked to manage todos, call the openOrCloseTodos frontend tool with open=true first. "
        "Keep responses to 1-2 sentences."
    ),
)
print("\u2713 Agent graph updated!")

## Phase 2: Frontend setup

`useAgent()` gives you a live handle to the agent's shared state: `agent.state.todos` reflects the current list, and `agent.setState({ todos: updated })` pushes changes back.

`useFrontendTool` registers a tool that runs in the browser — the agent calls it like any backend tool, but the handler executes client-side and can update React state directly.

### Utilize the frontend tools and shared state

In [ ]:
%%writefile frontend/src/App.tsx
import { z } from "zod"
import { CopilotChat, useAgent, useFrontendTool } from "@copilotkit/react-core/v2";
import { useState } from "react";
import { TodoAppLayout } from "@/components/todo-app-layout";
import { TodoList } from "@/components/todo-list";

export default function App() {
  const [todosOpen, setTodosOpen] = useState(false);

  // 🪁 Register a frontend tool the agent can call to control the UI
  useFrontendTool({
    name: "openOrCloseTodos",
    description: "Open or close the todo panel.",
    parameters: z.object({ open: z.boolean()}),
    handler: async ({open}) => {
      setTodosOpen(open);
      return `Todos are ${ open ? 'open' : 'closed'}.`;
    },
  });

  // 🪁 Subscribe to shared agent state
  const { agent } = useAgent();

  return (
    <TodoAppLayout
      chat={<CopilotChat />}
      open={todosOpen}
      onOpenChange={setTodosOpen}
      panel={(onClose) => (
        <TodoList
          // 🪁 Read shared state
          todos={agent.state.todos || []} 

          // 🪁 Write shared state
          onUpdate={(updated) => agent.setState({ todos: updated })}

          isRunning={agent.isRunning}
          onClose={onClose}
        />
      )}
    />
  );
}

### Give it a try! 

1. **"Add three todos about learning CopilotKit"** — the agent should open the panel and populate the list.
2. Check off a todo directly in the panel
3. Then ask **"What's on my list?"** — the agent reads the latest state.
4. **"Remove all completed todos"** — the agent calls `manage_todos` with the filtered list.

In [ ]:
from helper import display_app
display_app(port=3006)

## What you learned

- **Shared agent state** gives the backend and frontend a single source of truth — no manual sync needed.
- **`Command(update={...})`** lets a LangGraph tool update graph state and return a tool result in one step.
- **`useFrontendTool`** exposes browser-side behavior (like opening a panel) to the agent as a callable tool.
- **`useAgent`** + `agent.setState()` lets the user edit state directly from the UI, and the agent sees those edits on the next turn.

## Course wrap-up

Over six lessons you progressed from a basic agent chat to a fully synchronized fullstack app:

- **L2** — Connected a LangChain agent to a CopilotKit chat UI.
- **L3** — Registered controlled components so the agent can render charts and cards.
- **L4** — Used declarative A2UI schemas to separate data from presentation.
- **L5** — Hosted open-ended MCP Apps for tools like whiteboards and design surfaces.
- **L6** — Synchronized shared state between the agent and a React app with frontend tools.

These patterns compose. Explore the [CopilotKit docs](https://docs.copilotkit.ai) and the [AG-UI protocol spec](https://docs.ag-ui.com) to keep building.